# Linear Dependence and Independence

*Course notes for **Math for Machine Learning**, C1 · W1 · L1 — "Linear Dependence and Independence" (DeepLearning.AI).*

In [Singular vs Non-Singular Matrices](./C1_W1_L1_singular_vs_nonsingular_matrices.ipynb) we judged a matrix by setting its constants to 0. This lecture gives the *reason* a matrix is singular: its **rows carry redundant information**.

> **Linearly dependent rows** — at least one row can be built from the others (a multiple, a sum, an average, …). The matrix is **singular**.
>
> **Linearly independent rows** — no row can be built from the others; each adds new information. The matrix is **non-singular**.

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def analyze(A):
    """Report rank, determinant, and dependent/independent + singular/non-singular."""
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    rank = np.linalg.matrix_rank(A)
    det = np.linalg.det(A)
    independent = rank == n
    return {
        "rank": rank, "det": round(float(det), 3),
        "rows": "independent" if independent else "dependent",
        "matrix": "non-singular" if independent else "singular",
    }

## 1. Linear dependence between rows (2×2)

The two familiar systems, as coefficient matrices:

$$
\textbf{Non-singular}\;\begin{bmatrix} 1 & 1 \\ 1 & 2 \end{bmatrix}
\qquad\qquad
\textbf{Singular}\;\begin{bmatrix} 1 & 1 \\ 2 & 2 \end{bmatrix}
$$

- **Non-singular:** neither row is a multiple of the other → rows are **linearly independent**.
- **Singular:** row 2 $= 2 \times$ row 1 → rows are **linearly dependent**.

So *singular ⇔ dependent rows*, and *non-singular ⇔ independent rows*.

In [2]:
for name, A in {"Non-singular": [[1, 1], [1, 2]],
                "Singular":     [[1, 1], [2, 2]]}.items():
    r = analyze(A)
    print(f"{name:13s} {np.array(A).tolist()}  ->  rows {r['rows']}, matrix {r['matrix']}")

Non-singular  [[1, 1], [1, 2]]  ->  rows independent, matrix non-singular
Singular      [[1, 1], [2, 2]]  ->  rows dependent, matrix singular


## 2. Dependence is more than "a multiple" (3×3)

A row can depend on the others through *any* linear combination — sums and averages count too. Three examples (constants set to 0, so we look only at the rows):

**(a)** $\begin{bmatrix} 1&1&1 \\ 2&2&2 \\ 3&3&3 \end{bmatrix}$ — Row 2 $=2\times$Row 1, Row 3 $=3\times$Row 1. **Dependent → singular.**

**(b)** $\begin{bmatrix} 1&1&1 \\ 1&1&2 \\ 1&1&3 \end{bmatrix}$ — the **average** of Row 1 and Row 3 is Row 2 ($\tfrac{(1,1,1)+(1,1,3)}{2}=(1,1,2)$). **Dependent → singular.**

**(c)** $\begin{bmatrix} 1&1&1 \\ 1&2&1 \\ 1&1&2 \end{bmatrix}$ — no row is any combination of the others. **Independent → non-singular.**

In [3]:
examples = {
    "(a) multiples": [[1, 1, 1], [2, 2, 2], [3, 3, 3]],
    "(b) average":   [[1, 1, 1], [1, 1, 2], [1, 1, 3]],
    "(c) none":      [[1, 1, 1], [1, 2, 1], [1, 1, 2]],
}
for name, A in examples.items():
    r = analyze(A)
    print(f"{name:15s} det={r['det']:6.2f}  rank={r['rank']}/3  ->  rows {r['rows']:11s} ({r['matrix']})")

# Verify the hidden relation in (b): average of row 1 and row 3 equals row 2.
B = np.array(examples["(b) average"], dtype=float)
print("\n(b) (Row1 + Row3) / 2 =", (B[0] + B[2]) / 2, " == Row2", B[1])

(a) multiples   det=  0.00  rank=1/3  ->  rows dependent   (singular)
(b) average     det=  0.00  rank=2/3  ->  rows dependent   (singular)
(c) none        det=  1.00  rank=3/3  ->  rows independent (non-singular)

(b) (Row1 + Row3) / 2 = [1. 1. 2.]  == Row2 [1. 1. 2.]


## 3. Why it equals singularity

If a row is a linear combination of the others, it adds **no new constraint** to the system. With $n$ unknowns but fewer than $n$ truly independent equations, the homogeneous system $A\mathbf{x}=\mathbf{0}$ has non-trivial solutions — exactly the definition of a **singular** matrix. The number of independent rows is the matrix's **rank**:

$$ \text{rank}(A) = n \;\Leftrightarrow\; \text{independent rows} \;\Leftrightarrow\; \text{non-singular} \;\Leftrightarrow\; \det(A) \neq 0. $$

## 4. Exercise: classify these matrices

Determine whether each matrix has linearly **dependent** (singular) or **independent** (non-singular) rows:

$$
M_1=\begin{bmatrix}1&0&1\\0&1&0\\3&2&3\end{bmatrix}\quad
M_2=\begin{bmatrix}1&1&1\\1&1&2\\0&0&-1\end{bmatrix}\quad
M_3=\begin{bmatrix}1&1&1\\0&2&2\\0&0&3\end{bmatrix}\quad
M_4=\begin{bmatrix}1&2&5\\0&3&-2\\2&4&10\end{bmatrix}
$$

Relations to spot: $M_1$: $3\,\text{R1}+2\,\text{R2}=\text{R3}$ · $M_2$: $\text{R1}-\text{R2}=\text{R3}$ · $M_3$: none · $M_4$: $\text{R3}=2\,\text{R1}$.

In [4]:
problems = {
    "M1": [[1, 0, 1], [0, 1, 0], [3, 2, 3]],
    "M2": [[1, 1, 1], [1, 1, 2], [0, 0, -1]],
    "M3": [[1, 1, 1], [0, 2, 2], [0, 0, 3]],
    "M4": [[1, 2, 5], [0, 3, -2], [2, 4, 10]],
}
for name, A in problems.items():
    r = analyze(A)
    print(f"{name}: det={r['det']:7.2f}  rank={r['rank']}/3  ->  {r['rows']:11s} ({r['matrix']})")

M1: det=  -0.00  rank=2/3  ->  dependent   (singular)
M2: det=   0.00  rank=2/3  ->  dependent   (singular)
M3: det=   6.00  rank=3/3  ->  independent (non-singular)
M4: det=   0.00  rank=2/3  ->  dependent   (singular)


**Expected:** $M_1$ dependent (singular), $M_2$ dependent (singular), $M_3$ independent (non-singular), $M_4$ dependent (singular).

## 5. Takeaways

- **Dependent rows** (one is a linear combination of the others) ⇒ the matrix is **singular**.
- **Independent rows** (none is a combination of the others) ⇒ the matrix is **non-singular**.
- Dependence isn't only "a multiple of another row" — sums, averages, and any linear combination count.
- The count of independent rows is the **rank**; full rank ⇔ independent ⇔ non-singular ⇔ $\det \neq 0$.

*The same story holds for columns — a square matrix's rows are dependent exactly when its columns are.*